# Shared notebook adapters

This small bridge loads the teammates' **existing function definitions**, not their expensive demonstration/sweep cells. Tudor's engine has a quiet loading guard; Carla's named functions are extracted with Python AST (no copied algorithms). Errors remain visible. Alexandra's `objective_study.py` is retained because her notebook and experiment scripts import it. Alex's configuration is isolated in its own namespace. Run `all_together.ipynb` for the explained benchmark.

In [ ]:
from pathlib import Path
import ast
import itertools
import json
from collections import deque
import numpy as np
from scipy.optimize import minimize, OptimizeResult
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter, ParameterVector
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'bench' / 'qgd_bench.ipynb').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Run from the repository or one of its notebook directories.')

def notebook_cells(relative):
    return json.loads((ROOT / relative).read_text(encoding='utf-8'))['cells']

def load_definitions(relative, names):
    namespace = dict(globals())
    # Extracted functions do not need all_together's notebook-wide registry.
    namespace.setdefault('register', lambda *_args, **_kwargs: lambda function: function)
    found = set()
    for cell in notebook_cells(relative):
        if cell['cell_type'] != 'code':
            continue
        source = ''.join(cell['source'])
        tree = ast.parse(source)
        nodes = [node for node in tree.body
                 if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name in names]
        for node in nodes:
            if node.name in found:
                continue  # use the implemented lab, not its fallback appendix
            exec(compile(ast.Module(body=[node], type_ignores=[]), str(relative), 'exec'), namespace)
            found.add(node.name)
    if found != set(names):
        raise ValueError(f'Missing definitions in {relative}: {set(names) - found}')
    return namespace

def load_quiet_engine():
    namespace = {'QGD_QUIET': True, '__name__': 'qgd_shared_engine'}
    for cell in notebook_cells(Path('bench') / 'qgd_bench.ipynb'):
        if cell['cell_type'] == 'code':
            exec(compile(''.join(cell['source']), 'qgd_bench.ipynb', 'exec'), namespace)
    return namespace

tudor = load_quiet_engine()
carla = load_definitions(Path('bench') / 'all_together.ipynb', {
    'build_evidence', 'pairwise_cost', 'exact_reference', 'ising_from_evidence', 'ising_energy',
    'make_qaoa', 'warm_start_qaoa', 'make_ma_qaoa', 'train_ansatz',
    'edge_table', 'active_edges', 'components', 'n_blocks', 'prune', 'matrices_from_edges',
    'minima_reps', 'pruned_min_is_full_min'})
import sys
if str(ROOT / 'docs') not in sys.path:
    sys.path.insert(0, str(ROOT / 'docs'))
import objective_study as alexandra


## Local fitting and frozen replay

Carla's trainer fits the circuit's active parameters, so multi-angle circuits are not forced into a `2p` vector. If pruning removes every coupling, unused cost angles are set to zero in the returned full angle vector. The target energy array uses Qiskit little-endian basis order. Every fit is an exact **local statevector** computation. Alex's hardware setup is loaded only when requested and receives the same input/graph.

In [ ]:
def local_frozen_fit(circuit, params, energy, maxiter=60, seed=71):
    if len(set(params)) != len(params) or not set(circuit.parameters).issubset(params):
        raise ValueError('Circuit parameters and ordered parameter list disagree.')
    if len(energy) != 2 ** circuit.num_qubits:
        raise ValueError('Energy array must be in Qiskit basis-index order.')
    carla['energy_array'] = np.asarray(energy, dtype=float)
    active_indices = [i for i, param in enumerate(params) if param in circuit.parameters]
    active_params = [params[i] for i in active_indices]
    if not active_params:
        expected = float(Statevector.from_instruction(circuit).probabilities() @ energy)
        result = OptimizeResult(x=np.zeros(len(params)), fun=expected, nfev=1,
                                success=True, message='No active circuit parameters.')
        return circuit.copy(), result, 1
    measured, result, nfev = carla['train_ansatz'](
        circuit, active_params, maxiter=maxiter, n_starts=1, seed=seed)
    full_angles = np.zeros(len(params))
    full_angles[active_indices] = result.x
    result.x = full_angles
    frozen = measured.remove_final_measurements(inplace=False)
    if frozen.num_parameters:
        raise ValueError('Training did not freeze every circuit parameter.')
    return frozen, result, nfev

def load_hardware(reads, gammas, betas, evidence=None):
    namespace = {'hardware_inputs': dict(reads=reads, gammas=gammas, betas=betas, evidence=evidence)}
    for cell in notebook_cells(Path('docs') / 'hardware_experiments' / 'hardware_setup.ipynb'):
        if cell['cell_type'] == 'code':
            exec(compile(''.join(cell['source']), 'hardware_setup.ipynb', 'exec'), namespace)
    return namespace
